# MAJ (Memory Assisted Judge) on EvalsBench

This notebook evaluates MAJ on the EvalsBench dataset to compare:
1. **Stateless MAJ** - No memory
2. **Memory-Assisted MAJ** - Uses past judgments to improve accuracy

We'll compare against the EvalsBench leaderboard baselines.

## 1. Setup & Load Data

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load EvalsBench data
DATA_PATH = Path("data")

benchmark_df = pd.read_csv(DATA_PATH / "benchmark_df.csv")
annotation_df = pd.read_csv(DATA_PATH / "annotation_df.csv")

print(f"Benchmark samples: {len(benchmark_df)}")
print(f"Annotation samples: {len(annotation_df)}")
print(f"\nColumns: {benchmark_df.columns.tolist()}")

Benchmark samples: 160
Annotation samples: 24

Columns: ['topic', 'question', 'grading_notes', 'target', 'response', 'notes']


In [3]:
# Preview the data
benchmark_df.head()

,topic,question,grading_notes,target,response,notes
0,Pre-money valuation techniques,What are the key methods for determining the p...,"DCF method: !future cash flows!, requires proj...",pass,Determining the pre-money valuation of a tech ...,NaN
1,Pre-money valuation techniques,What are the key methods for determining the p...,"DCF method: !future cash flows!, requires proj...",fail,Determining the pre-money valuation of a tech ...,Removed the detailed explanation of 'Future Ca...
2,Post-money valuation metrics,What are the key factors to consider when calc...,"*Post-$ val* critical, pre-money val! + new in...",pass,When calculating the post-money valuation in a...,NaN
3,Post-money valuation metrics,What are the key factors to consider when calc...,"*Post-$ val* critical, pre-money val! + new in...",fail,When calculating the post-money valuation in a...,Dropped the explanation of 'Post-Money Valuati...
4,Convertible notes vs SAFE in seed funding,"When considering seed funding, what are the cr...","*Cap & discount*, maturity date (notes), inter...",pass,"When considering seed funding, startups often ...",NaN


In [4]:
# Check label distribution
print("Target distribution:")
print(benchmark_df['target'].value_counts())
print(f"\nPass rate: {(benchmark_df['target'] == 'pass').mean():.1%}")

Target distribution:
target
pass    80
fail    80
Name: count, dtype: int64

Pass rate: 50.0%


In [5]:
# Check topics
print(f"Unique topics: {benchmark_df['topic'].nunique()}")
print("\nTop 10 topics:")
benchmark_df['topic'].value_counts().head(10)

Unique topics: 80

Top 10 topics:


topic
Pre-money valuation techniques                          2
Post-money valuation metrics                            2
Real options analysis in startup investments            2
Syndication dynamics in venture capital                 2
Tech sector sensitivity to regulatory changes           2
Valuation discrepancies in private vs public markets    2
Capital efficiency metrics for startups                 2
Impact of public market trends on startup exits         2
Financial projections for hardware startups             2
Navigating antitrust issues in tech mergers             2
Name: count, dtype: int64

## 2. Explore Sample Structure

Each sample has:
- `question`: The task/question to answer
- `grading_notes`: Criteria for pass/fail
- `response`: The answer to judge
- `target`: Ground truth (pass/fail)

In [6]:
# Look at a PASS example
pass_example = benchmark_df[benchmark_df['target'] == 'pass'].iloc[0]

print("=" * 60)
print("PASS EXAMPLE")
print("=" * 60)
print(f"\nTOPIC: {pass_example['topic']}")
print(f"\nQUESTION:\n{pass_example['question']}")
print(f"\nGRADING NOTES:\n{pass_example['grading_notes']}")
print(f"\nRESPONSE (first 500 chars):\n{pass_example['response'][:500]}...")

PASS EXAMPLE

TOPIC: Pre-money valuation techniques

QUESTION:
What are the key methods for determining the pre-money valuation of a tech startup before a Series A investment round, and how do they differ?

GRADING NOTES:
DCF method: !future cash flows!, requires projections; Comp. analysis: similar co. multiples; VC method: rev x multiple - post-$; *Founder's share matter*; strategic buyers pay more.

RESPONSE (first 500 chars):
Determining the pre-money valuation of a tech startup before a Series A investment round is a critical step, as it significantly influences the negotiation process and the ultimate percentage of ownership acquired by new and existing shareholders. Here’s an overview of key valuation methods and their differences:

1. **Discounted Cash Flow (DCF) Method:**
   - **Overview:** This method focuses on forecasting the startup's future cash flows and discounting them back to their present value using a...


In [7]:
# Look at a FAIL example
fail_example = benchmark_df[benchmark_df['target'] == 'fail'].iloc[0]

print("=" * 60)
print("FAIL EXAMPLE")
print("=" * 60)
print(f"\nTOPIC: {fail_example['topic']}")
print(f"\nQUESTION:\n{fail_example['question']}")
print(f"\nGRADING NOTES:\n{fail_example['grading_notes']}")
print(f"\nRESPONSE (first 500 chars):\n{fail_example['response'][:500]}...")
print(f"\nNOTES (why it failed):\n{fail_example.get('notes', 'N/A')}")

FAIL EXAMPLE

TOPIC: Pre-money valuation techniques

QUESTION:
What are the key methods for determining the pre-money valuation of a tech startup before a Series A investment round, and how do they differ?

GRADING NOTES:
DCF method: !future cash flows!, requires projections; Comp. analysis: similar co. multiples; VC method: rev x multiple - post-$; *Founder's share matter*; strategic buyers pay more.

RESPONSE (first 500 chars):
Determining the pre-money valuation of a tech startup before a Series A investment round is a critical step, as it significantly influences the negotiation process and the ultimate percentage of ownership acquired by new and existing shareholders. Here’s an overview of key valuation methods and their differences:

1. **Discounted Cash Flow (DCF) Method:**
   - **Overview:** This method focuses on forecasting the startup's future cash flows and discounting them back to their present value using a...

NOTES (why it failed):
Removed the detailed explanation of 'F

## 3. Map EvalsBench to MAJ Format

| EvalsBench | MAJ |
|------------|-----|
| `question` + `grading_notes` | `task` |
| `response` | `agent_output` |
| `target` (pass/fail) | `is_successful` (true/false) |

In [8]:
def evalsbench_to_maj(row):
    """Convert EvalsBench row to MAJ format."""
    task = f"""Question: {row['question']}

Grading Criteria: {row['grading_notes']}"""
    
    return {
        'task': task,
        'agent_output': row['response'],
        'expected': row['target'] == 'pass',
        'topic': row['topic']
    }

# Test conversion
sample = evalsbench_to_maj(benchmark_df.iloc[0])
print("MAJ Format:")
print(f"Task: {sample['task'][:200]}...")
print(f"Expected: {sample['expected']}")

MAJ Format:
Task: Question: What are the key methods for determining the pre-money valuation of a tech startup before a Series A investment round, and how do they differ?

Grading Criteria: DCF method: !future cash flo...
Expected: True


## 4. Setup MAJ

In [9]:
# Install dependencies (uncomment for Colab)
# !pip install openai neo4j python-dotenv

In [10]:
import sys
sys.path.insert(0, 'src')

from judge import judge, judge_with_memory
from graph_manager import GraphManager

# Custom goal for Q&A evaluation
QA_GOAL = """Evaluate if the response adequately addresses the question based on the grading criteria.
The response should cover the KEY POINTS marked in the grading notes.
Focus on content accuracy and completeness, not writing style."""

print("MAJ loaded successfully!")

MAJ loaded successfully!


## 5. Run Benchmark

In [11]:
from tqdm import tqdm
import time

def run_benchmark(df, use_memory=False, gm=None, sample_size=None, model="gpt-4o-mini"):
    """
    Run MAJ on the benchmark dataset.
    
    Args:
        df: DataFrame with EvalsBench data
        use_memory: Whether to use memory-assisted judging
        gm: GraphManager instance (required if use_memory=True)
        sample_size: Number of samples to run (None = all)
        model: OpenAI model to use
    """
    if sample_size:
        df = df.sample(n=sample_size, random_state=42)
    
    results = []
    correct = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        sample = evalsbench_to_maj(row)
        
        try:
            if use_memory and gm:
                result = judge_with_memory(
                    task=sample['task'],
                    agent_output=sample['agent_output'],
                    graph_manager=gm,
                    goal=QA_GOAL,
                    model=model
                )
            else:
                result = judge(
                    task=sample['task'],
                    agent_output=sample['agent_output'],
                    goal=QA_GOAL,
                    model=model
                )
            
            predicted = result['attempt'].is_successful
            expected = sample['expected']
            is_correct = predicted == expected
            
            if is_correct:
                correct += 1
            
            results.append({
                'topic': sample['topic'],
                'expected': expected,
                'predicted': predicted,
                'correct': is_correct,
                'reasoning': result['attempt'].reasoning
            })
            
        except Exception as e:
            print(f"Error on sample {idx}: {e}")
            results.append({
                'topic': sample['topic'],
                'expected': sample['expected'],
                'predicted': None,
                'correct': False,
                'reasoning': str(e)
            })
    
    accuracy = correct / len(results) * 100
    print(f"\nAccuracy: {accuracy:.2f}% ({correct}/{len(results)})")
    
    return pd.DataFrame(results), accuracy

In [12]:
# Run on a small sample first to test
print("Testing on 10 samples (stateless)...")
test_results, test_acc = run_benchmark(benchmark_df, sample_size=10)
test_results

Testing on 10 samples (stateless)...


100%|██████████| 10/10 [00:43<00:00,  4.39s/it]


Accuracy: 50.00% (5/10)


,topic,expected,predicted,correct,reasoning
0,Financial projections for hardware startups,False,False,True,The response accurately covers some critical f...
1,Capital efficiency metrics for startups,True,True,True,The response thoroughly covers the key capital...
2,Valuation implications of tech stack choices,False,True,False,The response thoroughly addresses the question...
3,Impact of ESG criteria on startup evaluations,False,True,False,The output adequately covers all the key point...
4,Role of incubators and accelerators in tech ec...,True,True,True,The response comprehensively addresses the que...
5,Revenue forecasting methods for startups,False,True,False,The output effectively addresses the question ...
6,Assessing disruption potential in mature indus...,False,True,False,The response comprehensively addresses all key...
7,Differences between venture debt and equity fu...,False,True,False,The response adequately addresses the question...
8,Assessing disruption potential in mature indus...,True,True,True,The output comprehensively addresses all key p...
9,Use of special purpose acquisition companies (...,True,True,True,The response adequately addresses the question...


## 6. Full Benchmark: Stateless vs Memory

In [ ]:
# Run stateless benchmark (full dataset or subset)
SAMPLE_SIZE = 10  # Set to None for full dataset

print(f"Running STATELESS benchmark on {SAMPLE_SIZE or 'all'} samples...")
stateless_results, stateless_acc = run_benchmark(
    benchmark_df, 
    use_memory=False, 
    sample_size=SAMPLE_SIZE
)

In [ ]:
# Setup memory (Neo4j)
# For Colab, you can use Neo4j Aura free tier
gm = GraphManager(
    uri="bolt://localhost:7687",
    user="neo4j",
    password="password"  # Change this
)
gm.clear_all()  # Start fresh
print("Memory initialized!")

In [ ]:
# Run memory-assisted benchmark
# Memory builds organically as we judge

print(f"Running MEMORY-ASSISTED benchmark on {SAMPLE_SIZE or 'all'} samples...")

def run_organic_memory_benchmark(df, gm, sample_size=None, model="gpt-4o-mini"):
    """Run benchmark where memory builds organically."""
    if sample_size:
        df = df.sample(n=sample_size, random_state=42)
    
    results = []
    correct = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        sample = evalsbench_to_maj(row)
        
        try:
            result = judge_with_memory(
                task=sample['task'],
                agent_output=sample['agent_output'],
                graph_manager=gm,
                goal=QA_GOAL,
                model=model
            )
            
            predicted = result['attempt'].is_successful
            expected = sample['expected']
            is_correct = predicted == expected
            
            if is_correct:
                correct += 1
            
            # Store in memory for future judgments
            gm.store_policy(result['policy'])
            gm.store_attempt(result['attempt'])
            for issue in result['issues']:
                gm.store_issue(issue)
            for fix in result['fixes']:
                gm.store_fix(fix)
            for rel in result['relationships']:
                gm.create_relationship(rel['type'], rel['from_id'], rel['to_id'])
            
            # Store semantics
            semantic_rels = result.get('semantic_relationships', [])
            for i, semantic in enumerate(result.get('semantics', [])):
                if i < len(semantic_rels) and semantic_rels[i].get('is_new', True):
                    gm.get_or_create_semantic(semantic)
                if i < len(semantic_rels):
                    gm.create_relationship(
                        semantic_rels[i]['type'],
                        semantic_rels[i]['from_id'],
                        semantic_rels[i]['to_id']
                    )
            
            results.append({
                'topic': sample['topic'],
                'expected': expected,
                'predicted': predicted,
                'correct': is_correct,
                'memory_used': result.get('memory_used', {}),
                'reasoning': result['attempt'].reasoning
            })
            
        except Exception as e:
            print(f"Error on sample {idx}: {e}")
            results.append({
                'topic': sample['topic'],
                'expected': sample['expected'],
                'predicted': None,
                'correct': False,
                'reasoning': str(e)
            })
    
    accuracy = correct / len(results) * 100
    print(f"\nAccuracy: {accuracy:.2f}% ({correct}/{len(results)})")
    
    return pd.DataFrame(results), accuracy

memory_results, memory_acc = run_organic_memory_benchmark(
    benchmark_df, 
    gm, 
    sample_size=SAMPLE_SIZE
)

## 7. Results Comparison

In [ ]:
print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"\nStateless MAJ:       {stateless_acc:.2f}%")
print(f"Memory-Assisted MAJ: {memory_acc:.2f}%")
print(f"Improvement:         {memory_acc - stateless_acc:+.2f}%")
print("\n" + "=" * 60)
print("EvalsBench Leaderboard (gpt-4o-mini):")
print("=" * 60)
print("Vanilla:             84.49%")
print("Fixed Few-Shot:      79.79%")
print("Random Few-Shot:     75.86%")
print("Dynamic Few-Shot:    76.65%")
print("Auto Prompt Opt:     51.38%")

In [ ]:
# Error analysis by topic
print("\nAccuracy by Topic (Memory-Assisted):")
topic_acc = memory_results.groupby('topic')['correct'].mean().sort_values()
print("\nWorst 5 topics:")
print(topic_acc.head())
print("\nBest 5 topics:")
print(topic_acc.tail())

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Filter out errors
valid = memory_results[memory_results['predicted'].notna()]

cm = confusion_matrix(valid['expected'], valid['predicted'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fail', 'Pass'],
            yticklabels=['Fail', 'Pass'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('MAJ Confusion Matrix')
plt.show()

print("\nClassification Report:")
print(classification_report(valid['expected'], valid['predicted'], 
                          target_names=['Fail', 'Pass']))

## 8. Save Results

In [ ]:
# Save results
stateless_results.to_csv('maj_stateless_results.csv', index=False)
memory_results.to_csv('maj_memory_results.csv', index=False)

print("Results saved!")

In [ ]:
# Look at the test results to understand what's going wrong
test_results

In [13]:
# Let's look at a specific FAIL case that we got wrong
fail_case = benchmark_df.iloc[test_results[~test_results['correct']].index[0]]

print("=" * 60)
print("CASE WE GOT WRONG (expected FAIL, predicted PASS)")
print("=" * 60)
print(f"\nTOPIC: {fail_case['topic']}")
print(f"\nQUESTION:\n{fail_case['question']}")
print(f"\nGRADING NOTES:\n{fail_case['grading_notes']}")
print(f"\nNOTES (why it should fail):\n{fail_case.get('notes', 'N/A')}")
print(f"\nRESPONSE (first 800 chars):\n{fail_case['response'][:800]}...")

CASE WE GOT WRONG (expected FAIL, predicted PASS)

TOPIC: Post-money valuation metrics

QUESTION:
What are the key factors to consider when calculating post-money valuation in a startup financing round?

GRADING NOTES:
*Post-$ val* critical, pre-money val! + new invstmnt, % equity sold, fully-diluted shares calc, impact on *option pool*; Caution: anti-dilution, pref terms.

NOTES (why it should fail):
nan

RESPONSE (first 800 chars):
When calculating the post-money valuation in a startup financing round, it is critical to understand a few fundamental components that collectively determine how the valuation can accurately reflect the company’s worth and future potential. Here are the key factors to consider:

1. **Post-Money Valuation**: This is the critical value of the company immediately after the financing round is concluded. It’s a sum of the pre-money valuation and the new investment amount. Essentially, this value represents the company's worth after it has received fresh capital

In [14]:
# Let's see a proper FAIL case with notes
fail_with_notes = benchmark_df[(benchmark_df['target'] == 'fail') & (benchmark_df['notes'].notna())].iloc[0]

print("=" * 60)
print("FAIL CASE WITH NOTES")
print("=" * 60)
print(f"\nGRADING NOTES:\n{fail_with_notes['grading_notes']}")
print(f"\nNOTES (why it failed):\n{fail_with_notes['notes']}")

FAIL CASE WITH NOTES

GRADING NOTES:
DCF method: !future cash flows!, requires projections; Comp. analysis: similar co. multiples; VC method: rev x multiple - post-$; *Founder's share matter*; strategic buyers pay more.

NOTES (why it failed):
Removed the detailed explanation of 'Future Cash Flows' from the DCF method and the mention of 'Post-Money' calculations and Founder's share importance from the Venture Capital method. These omissions maintain the answer's coherence while strategically dropping essential points.


In [15]:
# Update the QA_GOAL to be stricter about key points
QA_GOAL = """Evaluate if the response covers ALL required key points from the grading criteria.

GRADING NOTATION:
- Text marked with !exclamation marks! = MUST be explicitly covered (critical requirement)
- Text marked with *asterisks* = Important points that should be addressed
- Other text = Supporting criteria

EVALUATION RULES:
1. If ANY !required! point is MISSING or only superficially mentioned → FAIL
2. If important *starred* points are missing → likely FAIL
3. The response must demonstrate understanding of the specific concepts, not just mention keywords
4. A well-written response that misses key points should still FAIL

Be STRICT: Missing even ONE required point means the response fails."""

print("Updated QA_GOAL")
print(QA_GOAL)

Updated QA_GOAL
Evaluate if the response covers ALL required key points from the grading criteria.

GRADING NOTATION:
- Text marked with !exclamation marks! = MUST be explicitly covered (critical requirement)
- Text marked with *asterisks* = Important points that should be addressed
- Other text = Supporting criteria

EVALUATION RULES:
1. If ANY !required! point is MISSING or only superficially mentioned → FAIL
2. If important *starred* points are missing → likely FAIL
3. The response must demonstrate understanding of the specific concepts, not just mention keywords
4. A well-written response that misses key points should still FAIL

Be STRICT: Missing even ONE required point means the response fails.


In [16]:
# Re-run the test with the stricter goal
print("Testing on 10 samples with STRICT goal...")
test_results2, test_acc2 = run_benchmark(benchmark_df, sample_size=10)
print(f"\nPrevious accuracy: 50%")
print(f"New accuracy: {test_acc2:.1f}%")

Testing on 10 samples with STRICT goal...


100%|██████████| 10/10 [01:01<00:00,  6.15s/it]


Accuracy: 60.00% (6/10)

Previous accuracy: 50%
New accuracy: 60.0%


In [17]:
# Check what's still wrong
test_results2

,topic,expected,predicted,correct,reasoning
0,Financial projections for hardware startups,False,False,True,The response fails to cover all required key p...
1,Capital efficiency metrics for startups,True,False,False,The response fails to explicitly mention the c...
2,Valuation implications of tech stack choices,False,False,True,The response fails to explicitly cover the req...
3,Impact of ESG criteria on startup evaluations,False,False,True,The response is missing several critical point...
4,Role of incubators and accelerators in tech ec...,True,False,False,The response is well-written and covers many p...
5,Revenue forecasting methods for startups,False,False,True,The response fails to explicitly cover the cri...
6,Assessing disruption potential in mature indus...,False,False,True,The response fails to explicitly address the c...
7,Differences between venture debt and equity fu...,False,False,True,The response fails to explicitly cover all req...
8,Assessing disruption potential in mature indus...,True,False,False,The response is well-structured and covers man...
9,Use of special purpose acquisition companies (...,True,False,False,The output is thorough and covers many aspects...


In [18]:
# Balance the goal - be strict on required points but fair on interpretation
QA_GOAL = """Evaluate if the response adequately covers the key points from the grading criteria.

GRADING NOTATION:
- Text marked with !exclamation marks! = MUST be covered (critical requirement)
- Text marked with *asterisks* = Important points that should be addressed
- Semicolons separate distinct criteria

EVALUATION RULES:
1. Required (!text!) points: Must be explicitly addressed with substantive explanation
2. Important (*text*) points: Should be covered, minor omissions acceptable if others are strong
3. A response can PASS if it covers the CORE concepts even if wording differs from criteria
4. Look for SUBSTANCE over exact keyword matching
5. FAIL only if critical points are genuinely missing or misunderstood

Be FAIR: A comprehensive response covering the main concepts should PASS."""

print("Balanced QA_GOAL")

# Re-run
print("\nTesting on 10 samples with BALANCED goal...")
test_results3, test_acc3 = run_benchmark(benchmark_df, sample_size=10)
print(f"\nOriginal (lenient): 50%")
print(f"Strict: 60%")
print(f"Balanced: {test_acc3:.1f}%")

Balanced QA_GOAL

Testing on 10 samples with BALANCED goal...


100%|██████████| 10/10 [00:36<00:00,  3.61s/it]


Accuracy: 50.00% (5/10)

Original (lenient): 50%
Strict: 60%
Balanced: 50.0%


In [19]:
# Check the pattern
test_results3[['expected', 'predicted', 'correct']]

,expected,predicted,correct
0,False,False,True
1,True,True,True
2,False,True,False
3,False,True,False
4,True,True,True
5,False,True,False
6,False,True,False
7,False,True,False
8,True,True,True
9,True,True,True


In [20]:
import re

def parse_grading_notes(notes):
    """Extract required (!) and important (*) points from grading notes."""
    required = re.findall(r'!([^!]+)!', notes)
    important = re.findall(r'\*([^*]+)\*', notes)
    return required, important

# Test on a sample
sample_notes = "DCF method: !future cash flows!, requires projections; Comp. analysis: similar co. multiples; VC method: rev x multiple - post-$; *Founder's share matter*; strategic buyers pay more."
req, imp = parse_grading_notes(sample_notes)
print(f"Required (!): {req}")
print(f"Important (*): {imp}")

Required (!): ['future cash flows']
Important (*): ["Founder's share matter"]


In [21]:
# New approach: explicitly list what must be checked
def evalsbench_to_maj_v2(row):
    """Convert EvalsBench row to MAJ format with parsed criteria."""
    required, important = parse_grading_notes(row['grading_notes'])
    
    task = f"""Question: {row['question']}

GRADING CRITERIA: {row['grading_notes']}

REQUIRED POINTS (must be covered):
{chr(10).join(f'- {r}' for r in required) if required else '- None explicitly marked'}

IMPORTANT POINTS (should be covered):
{chr(10).join(f'- {i}' for i in important) if important else '- None explicitly marked'}"""
    
    return {
        'task': task,
        'agent_output': row['response'],
        'expected': row['target'] == 'pass',
        'topic': row['topic']
    }

# Update QA_GOAL
QA_GOAL = """Evaluate if the response covers the required and important points.

DECISION RULES:
1. Check each REQUIRED POINT - if ANY is missing or only superficially mentioned → FAIL
2. Check IMPORTANT POINTS - missing these weakens the response
3. The response must SUBSTANTIVELY address each point, not just mention keywords

IMPORTANT: The grading criteria use shorthand notation. A point is "covered" if the response explains the concept, even if exact wording differs.

Return FAIL if required points are missing. Return PASS only if all required points are adequately covered."""

print("New approach: Explicit point extraction")
print(evalsbench_to_maj_v2(benchmark_df.iloc[0])['task'][:500])

New approach: Explicit point extraction
Question: What are the key methods for determining the pre-money valuation of a tech startup before a Series A investment round, and how do they differ?

GRADING CRITERIA: DCF method: !future cash flows!, requires projections; Comp. analysis: similar co. multiples; VC method: rev x multiple - post-$; *Founder's share matter*; strategic buyers pay more.

REQUIRED POINTS (must be covered):
- future cash flows

IMPORTANT POINTS (should be covered):
- Founder's share matter


In [22]:
# Update the benchmark function to use v2 converter
def run_benchmark_v2(df, use_memory=False, gm=None, sample_size=None, model="gpt-4o-mini"):
    """Run MAJ on the benchmark dataset with explicit point extraction."""
    if sample_size:
        df = df.sample(n=sample_size, random_state=42)
    
    results = []
    correct = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        sample = evalsbench_to_maj_v2(row)  # Use v2 converter
        
        try:
            result = judge(
                task=sample['task'],
                agent_output=sample['agent_output'],
                goal=QA_GOAL,
                model=model
            )
            
            predicted = result['attempt'].is_successful
            expected = sample['expected']
            is_correct = predicted == expected
            
            if is_correct:
                correct += 1
            
            results.append({
                'topic': sample['topic'],
                'expected': expected,
                'predicted': predicted,
                'correct': is_correct,
                'reasoning': result['attempt'].reasoning
            })
            
        except Exception as e:
            print(f"Error on sample {idx}: {e}")
            results.append({
                'topic': sample['topic'],
                'expected': sample['expected'],
                'predicted': None,
                'correct': False,
                'reasoning': str(e)
            })
    
    accuracy = correct / len(results) * 100
    print(f"\nAccuracy: {accuracy:.2f}% ({correct}/{len(results)})")
    
    return pd.DataFrame(results), accuracy

# Test with explicit point extraction
print("Testing with EXPLICIT POINT EXTRACTION...")
test_results4, test_acc4 = run_benchmark_v2(benchmark_df, sample_size=10)
print(f"\nOriginal: 50%")
print(f"Strict: 60%") 
print(f"Balanced: 50%")
print(f"Explicit points: {test_acc4:.1f}%")

Testing with EXPLICIT POINT EXTRACTION...


100%|██████████| 10/10 [00:53<00:00,  5.35s/it]


Accuracy: 90.00% (9/10)

Original: 50%
Strict: 60%
Balanced: 50%
Explicit points: 90.0%


In [23]:
test_results4[['expected', 'predicted', 'correct']]

,expected,predicted,correct
0,False,False,True
1,True,True,True
2,False,False,True
3,False,False,True
4,True,False,False
5,False,False,True
6,False,False,True
7,False,False,True
8,True,True,True
9,True,True,True


In [24]:
# Run MAJ WITH MEMORY on 10 samples
from graph_manager import GraphManager

# Setup Neo4j
gm = GraphManager(
    uri="bolt://localhost:7687",
    user="neo4j", 
    password="password"
)
gm.clear_all()  # Start fresh
print("Memory initialized (Neo4j cleared)\n")

# Update organic memory benchmark to use v2 converter
def run_organic_memory_benchmark_v2(df, gm, sample_size=None, model="gpt-4o-mini"):
    """Run benchmark where memory builds organically with explicit point extraction."""
    if sample_size:
        df = df.sample(n=sample_size, random_state=42)
    
    results = []
    correct = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        sample = evalsbench_to_maj_v2(row)  # Use v2 converter with explicit points
        
        try:
            result = judge_with_memory(
                task=sample['task'],
                agent_output=sample['agent_output'],
                graph_manager=gm,
                goal=QA_GOAL,
                model=model
            )
            
            predicted = result['attempt'].is_successful
            expected = sample['expected']
            is_correct = predicted == expected
            
            if is_correct:
                correct += 1
            
            # Store in memory
            gm.store_policy(result['policy'])
            gm.store_attempt(result['attempt'])
            for issue in result['issues']:
                gm.store_issue(issue)
            for fix in result['fixes']:
                gm.store_fix(fix)
            for rel in result['relationships']:
                gm.create_relationship(rel['type'], rel['from_id'], rel['to_id'])
            
            # Store semantics
            semantic_rels = result.get('semantic_relationships', [])
            for i, semantic in enumerate(result.get('semantics', [])):
                if i < len(semantic_rels) and semantic_rels[i].get('is_new', True):
                    gm.get_or_create_semantic(semantic)
                if i < len(semantic_rels):
                    gm.create_relationship(
                        semantic_rels[i]['type'],
                        semantic_rels[i]['from_id'],
                        semantic_rels[i]['to_id']
                    )
            
            results.append({
                'topic': sample['topic'],
                'expected': expected,
                'predicted': predicted,
                'correct': is_correct,
                'memory_used': result.get('memory_used', {}),
                'reasoning': result['attempt'].reasoning
            })
            
        except Exception as e:
            print(f"Error on sample {idx}: {e}")
            results.append({
                'topic': sample['topic'],
                'expected': sample['expected'],
                'predicted': None,
                'correct': False,
                'reasoning': str(e)
            })
    
    accuracy = correct / len(results) * 100
    print(f"\nAccuracy: {accuracy:.2f}% ({correct}/{len(results)})")
    
    return pd.DataFrame(results), accuracy

print("Running MAJ WITH MEMORY on 10 samples...")
results_mem_10, acc_mem_10 = run_organic_memory_benchmark_v2(benchmark_df, gm, sample_size=10)

print(f"\n" + "=" * 60)
print(f"RESULTS (10 samples)")
print("=" * 60)
print(f"MAJ Stateless:   90.0%")
print(f"MAJ With Memory: {acc_mem_10:.1f}%")
print(f"Vanilla baseline: 84.49%")
print("=" * 60)

TypeError: GraphManager.__init__() got an unexpected keyword argument 'uri'

In [25]:
# GraphManager uses env vars, not constructor args
gm = GraphManager()
gm.clear_all()
print("Memory initialized (Neo4j cleared)\n")

print("Running MAJ WITH MEMORY on 10 samples...")
results_mem_10, acc_mem_10 = run_organic_memory_benchmark_v2(benchmark_df, gm, sample_size=10)

print(f"\n" + "=" * 60)
print(f"RESULTS (10 samples)")
print("=" * 60)
print(f"MAJ Stateless:   90.0%")
print(f"MAJ With Memory: {acc_mem_10:.1f}%")
print(f"Vanilla baseline: 84.49%")
print("=" * 60)

Memory initialized (Neo4j cleared)

Running MAJ WITH MEMORY on 10 samples...


NameError: name 'run_organic_memory_benchmark_v2' is not defined

In [26]:
# Define the function first
def run_organic_memory_benchmark_v2(df, gm, sample_size=None, model="gpt-4o-mini"):
    """Run benchmark where memory builds organically with explicit point extraction."""
    if sample_size:
        df = df.sample(n=sample_size, random_state=42)
    
    results = []
    correct = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        sample = evalsbench_to_maj_v2(row)
        
        try:
            result = judge_with_memory(
                task=sample['task'],
                agent_output=sample['agent_output'],
                graph_manager=gm,
                goal=QA_GOAL,
                model=model
            )
            
            predicted = result['attempt'].is_successful
            expected = sample['expected']
            is_correct = predicted == expected
            
            if is_correct:
                correct += 1
            
            # Store in memory
            gm.store_policy(result['policy'])
            gm.store_attempt(result['attempt'])
            for issue in result['issues']:
                gm.store_issue(issue)
            for fix in result['fixes']:
                gm.store_fix(fix)
            for rel in result['relationships']:
                gm.create_relationship(rel['type'], rel['from_id'], rel['to_id'])
            
            # Store semantics
            semantic_rels = result.get('semantic_relationships', [])
            for i, semantic in enumerate(result.get('semantics', [])):
                if i < len(semantic_rels) and semantic_rels[i].get('is_new', True):
                    gm.get_or_create_semantic(semantic)
                if i < len(semantic_rels):
                    gm.create_relationship(
                        semantic_rels[i]['type'],
                        semantic_rels[i]['from_id'],
                        semantic_rels[i]['to_id']
                    )
            
            results.append({
                'topic': sample['topic'],
                'expected': expected,
                'predicted': predicted,
                'correct': is_correct,
                'memory_used': result.get('memory_used', {}),
                'reasoning': result['attempt'].reasoning
            })
            
        except Exception as e:
            print(f"Error on sample {idx}: {e}")
            results.append({
                'topic': sample['topic'],
                'expected': sample['expected'],
                'predicted': None,
                'correct': False,
                'reasoning': str(e)
            })
    
    accuracy = correct / len(results) * 100
    print(f"\nAccuracy: {accuracy:.2f}% ({correct}/{len(results)})")
    
    return pd.DataFrame(results), accuracy

# Now run it
print("Running MAJ WITH MEMORY on 10 samples...")
results_mem_10, acc_mem_10 = run_organic_memory_benchmark_v2(benchmark_df, gm, sample_size=10)

print(f"\n" + "=" * 60)
print(f"RESULTS (10 samples)")
print("=" * 60)
print(f"MAJ Stateless:   90.0%")
print(f"MAJ With Memory: {acc_mem_10:.1f}%")
print(f"Vanilla baseline: 84.49%")
print("=" * 60)

Running MAJ WITH MEMORY on 10 samples...


 10%|█         | 1/10 [00:10<01:32, 10.24s/it]

Error on sample 105: 'GraphManager' object has no attribute 'store_policy'


 20%|██        | 2/10 [00:20<01:22, 10.34s/it]

Error on sample 108: 'GraphManager' object has no attribute 'store_policy'


 30%|███       | 3/10 [00:30<01:11, 10.18s/it]

Error on sample 141: 'GraphManager' object has no attribute 'store_policy'


 30%|███       | 3/10 [00:44<01:42, 14.71s/it]


KeyboardInterrupt: 